# How the MLP block can store a fact (toy AND gate, then GPT-2 small in NumPy)
1. Toy: 5 perpendicular directions (Michael, Jordan, Phelps, Alexis, basketball); one neuron with row M + J, bias -1, ReLU, column = basketball.
2. GPT-2 small: which athlete -> sport facts does it get right?
3. Zero one MLP at a time (all positions, or only the last name token) and measure the drop.
4. Exact split of the final logit difference (basketball minus football) into the parts each block wrote.
5. The neurons that write it: their output rows read through the unembedding (Geva et al. 2021, §4).
Weights: `tools/gpt2_small.py` (first run downloads about 500 MB to ~/.cache/campusx/gpt2).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parents[2] / "tools"))
import gpt2_small as g

np.set_printoptions(precision=3, suppress=True)
DATA = Path("data"); DATA.mkdir(exist_ok=True)

## 1. Toy AND gate
Directions are the 5 standard basis vectors of a 5-number space, so they are exactly perpendicular.

In [ ]:
names = ["Michael", "Jordan", "Phelps", "Alexis", "basketball"]
M, J, P, A, Bb = np.eye(5)
row, bias, col = M + J, -1.0, Bb                       # W_up row, b_up entry, W_down column
inputs = {"Michael Jordan": M + J, "Michael Phelps": M + P, "Alexis Jordan": A + J, "Michael": M, "Phelps": P}
rows = []
for name, e in inputs.items():
    dot = row @ e
    pre = dot + bias
    relu = max(pre, 0.0)
    out = e + relu * col
    rows.append(dict(input=name, row_dot_e=dot, plus_bias=pre, after_relu=relu, gelu=float(g.gelu(np.array(pre))),
                     basketball_before=e @ Bb, basketball_after=out @ Bb, basketball_if_no_relu=(e + pre * col) @ Bb))
toy = pd.DataFrame(rows)
toy.to_csv(DATA / "toy_and_gate.csv", index=False)
print(toy.round(3).to_string(index=False))

## 2. GPT-2 small: athlete -> sport

In [ ]:
W = g.load()
WE = W["wte.weight"]
SPORTS = [" basketball", " football", " golf", " hockey", " tennis", " baseball", " soccer", " cricket", " boxing", " swimming"]
sid = {s: g.encode(s)[0] for s in SPORTS}
assert all(len(g.encode(s)) == 1 for s in SPORTS)
athletes = {"Michael Jordan": " basketball", "LeBron James": " basketball", "Kobe Bryant": " basketball",
            "Shaquille O'Neal": " basketball", "Tiger Woods": " golf", "Phil Mickelson": " golf",
            "Wayne Gretzky": " hockey", "Sidney Crosby": " hockey", "Serena Williams": " tennis",
            "Roger Federer": " tennis", "Rafael Nadal": " tennis", "Babe Ruth": " baseball", "Derek Jeter": " baseball",
            "Tom Brady": " football", "Peyton Manning": " football", "Michael Phelps": " swimming",
            "Muhammad Ali": " boxing", "Sachin Tendulkar": " cricket", "Lionel Messi": " football"}
rows = []
for name, sport in athletes.items():
    ids = g.encode(f"{name} plays the sport of")
    p = g.softmax(g.forward(ids, W)["logits"][-1])
    top = int(p.argmax())
    sp = {s: p[i] for s, i in sid.items()}
    best_sport = max(sp, key=sp.get)
    rows.append(dict(athlete=name, true=sport.strip(), top1=g.decode([top]).strip(), p_top1=p[top], p_true=p[sid[sport]],
                     rank_true=int((p > p[sid[sport]]).sum()) + 1, best_of_10_sports=best_sport.strip()))
facts = pd.DataFrame(rows)
facts["correct_top1"] = facts.top1 == facts.true
facts.to_csv(DATA / "athletes.csv", index=False)
print(facts.round(3).to_string(index=False))
print("top-1 correct:", facts.correct_top1.sum(), "of", len(facts), "; true sport best among the 10 sports:",
      (facts.best_of_10_sports == facts.true).sum())

## 3. Zero one MLP at a time
Zero-ablation: replace a layer's MLP output by 0 (at every position, or only at the last token of the name), run the rest unchanged.
Main example: "Michael Jordan plays the sport of" (Sanderson 2024, Ch 7; Nanda et al. 2023).

In [ ]:
def prob(ids, target, hook=None):
    return float(g.softmax(g.forward(ids, W, hook)["logits"][-1])[target])

def ablate(kind, layer, pos=None):
    def hook(name, l, x):
        if name == kind and l == layer:
            x = x.copy()
            if pos is None: x[:] = 0
            else: x[pos] = 0
        return x
    return hook

name_last = {}
for name in athletes:
    n_name = len(g.encode(name))                      # the name's tokens come first in the prompt
    name_last[name] = n_name - 1
main = "Michael Jordan"
ids = g.encode(f"{main} plays the sport of")
print("tokens:", [g.decode([i]) for i in ids], " last name token index:", name_last[main])
base = prob(ids, sid[" basketball"])
rows = []
for l in range(12):
    rows.append(dict(layer=l + 1, base=base,
                     mlp_all=prob(ids, sid[" basketball"], ablate("mlp_out", l)),
                     mlp_name=prob(ids, sid[" basketball"], ablate("mlp_out", l, name_last[main])),
                     attn_all=prob(ids, sid[" basketball"], ablate("attn_out", l))))
abl = pd.DataFrame(rows)
abl.to_csv(DATA / "ablation_jordan.csv", index=False)
print(f"p(basketball) = {base:.3f}")
print(abl.round(3).to_string(index=False))

### Which sport? Group ablations on all athletes GPT-2 small gets right
Measure: is the true sport still ranked first among the 10 sports, and its share of the 10 sports' probability.
This separates "which sport" (the fact) from "a sport comes next" (the question). Name tokens = the athlete's tokens at the start of the prompt.

In [ ]:
good = facts[facts.correct_top1].athlete.tolist()
sp_ids = [sid[s] for s in SPORTS]
def sport_metrics(ids_a, t, hook=None):
    p = g.softmax(g.forward(ids_a, W, hook)["logits"][-1])
    q = p[sp_ids]; k = SPORTS.index(t)
    return dict(p_true=float(p[sid[t]]), share=float(q[k] / q.sum()), still_first=bool(q.argmax() == k))

def group(layers, where):
    def hook(name, l, x):
        if name == "mlp_out" and l in layers:
            x = x.copy(); x[where] = 0
        return x
    return hook

conds = {"nothing zeroed": (None, None), "MLP 1, name tokens": ([0], "name"), "MLPs 2-6, name tokens": (range(1, 6), "name"),
         "MLPs 2-6, other tokens": (range(1, 6), "rest"), "MLPs 7-12, name tokens": (range(6, 12), "name"),
         "MLPs 7-12, other tokens": (range(6, 12), "rest")}
rows = []
for name in good:
    ids_a = g.encode(f"{name} plays the sport of"); n = len(g.encode(name))
    pos = {"name": list(range(n)), "rest": list(range(n, len(ids_a)))}
    for c, (layers, where) in conds.items():
        hook = None if layers is None else group(list(layers), pos[where])
        rows.append(dict(athlete=name, condition=c, **sport_metrics(ids_a, athletes[name], hook)))
grp = pd.DataFrame(rows)
grp.to_csv(DATA / "ablation_groups.csv", index=False)
summary = grp.groupby("condition", sort=False).agg(still_first=("still_first", "sum"), mean_share=("share", "mean"),
                                                   mean_p_true=("p_true", "mean"))
summary["athletes"] = len(good)
summary.to_csv(DATA / "ablation_summary.csv")
print(summary.round(3).to_string())
print(grp[grp.condition == "MLPs 2-6, name tokens"][["athlete", "still_first", "share"]].round(2).to_string(index=False))

## 4. Who writes "basketball rather than football"? An exact split of the logit difference
With the final LayerNorm's scale s frozen at its value in this run, ln_f(x) = gamma * (x - mean(x)) / s + beta is linear in x.
The stream is a sum of parts (token + position embedding, 12 attention outputs, 12 MLP outputs), so the logit difference
logit(basketball) - logit(football) splits exactly into one term per part (plus a constant from beta).

In [ ]:
out = g.forward(ids, W)
x = out["resid"][-1, -1]
s = np.sqrt(((x - x.mean()) ** 2).mean() + 1e-5)
gam, bet = W["ln_f.weight"], W["ln_f.bias"]
u = WE[sid[" basketball"]] - WE[sid[" football"]]
def part(v):                                             # contribution of one additive part of the stream
    return float((gam * (v - v.mean()) / s) @ u)
parts = {"embedding": out["resid"][0, -1]}
for l in range(12):
    parts[f"attn {l+1}"] = out["attn_out"][l, -1]
    parts[f"mlp {l+1}"] = out["mlp_out"][l, -1]
contrib = {k: part(v) for k, v in parts.items()}
const = float(bet @ u)
total = sum(contrib.values()) + const
direct = out["logits"][-1, sid[" basketball"]] - out["logits"][-1, sid[" football"]]
print(f"logit difference: {direct:.3f}; sum of parts + beta term: {total:.3f}")
assert abs(total - direct) < 1e-2
dla = pd.DataFrame({"part": list(contrib), "contribution": list(contrib.values())})
dla.to_csv(DATA / "logit_split_jordan.csv", index=False)
print(dla.round(3).to_string(index=False))
print("beta term:", round(const, 3), "; all MLPs:", round(sum(v for k, v in contrib.items() if k.startswith('mlp')), 3),
      "; all attention:", round(sum(v for k, v in contrib.items() if k.startswith('attn')), 3))

The same split at the last *name* token " Jordan" (before the prompt asks about sport): how much basketball-over-football do the MLPs there already write?

In [ ]:
pn = name_last[main]
x_n = out["resid"][-1, pn]; s_n = np.sqrt(((x_n - x_n.mean()) ** 2).mean() + 1e-5)
contrib_n = {f"mlp {l+1}": float((gam * (out['mlp_out'][l, pn] - out['mlp_out'][l, pn].mean()) / s_n) @ u) for l in range(12)}
contrib_n.update({f"attn {l+1}": float((gam * (out['attn_out'][l, pn] - out['attn_out'][l, pn].mean()) / s_n) @ u) for l in range(12)})
print({k: round(v, 2) for k, v in contrib_n.items() if k.startswith("mlp")})

## 5. The neurons: which ones write the basketball-over-football direction at the last position?
Neuron i of layer l writes act_i * (row i of c_proj). Its share: act_i * (c_proj[i] read through ln_f and u).

In [ ]:
best_layers = dla[dla.part.str.startswith("mlp")].sort_values("contribution", ascending=False).head(3).part.tolist()
print("MLP layers with the largest contributions:", best_layers)
rows = []
for name_l in best_layers:
    l = int(name_l.split()[1]) - 1
    act = out["mlp_act"][l, -1]
    Wd = W[f"h.{l}.mlp.c_proj.weight"]                   # (3072, 768): row i = what neuron i writes
    per = act * (((Wd - Wd.mean(1, keepdims=True)) * gam / s) @ u)
    for i in np.argsort(-per)[:3]:
        logit_row = WE @ Wd[i]                            # Geva et al. 2021, §4: read the written vector with the output embedding
        rows.append(dict(layer=l + 1, neuron=int(i), activation=float(act[i]), contribution=float(per[i]),
                         share_of_layer=float(per[i] / per.sum()),
                         top_tokens=" | ".join(g.decode([t]).strip() for t in np.argsort(-logit_row)[:8])))
    print(f"layer {l+1}: {int((act > 0).sum())} of 3072 neurons positive; top 10 neurons give {np.sort(per)[::-1][:10].sum() / per.sum():.0%} of the layer's contribution")
neur = pd.DataFrame(rows)
neur.to_csv(DATA / "top_neurons.csv", index=False)
print(neur.round(3).to_string(index=False))